In [ ]:
import glob, os, random, pydicom
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
from pathlib import Path

# Configure pandas to display full, untruncated column contents
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", None)

report_col = "Report"
pid_col = "StudyInstanceUID"

DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
train = pd.read_csv(DATA_DIR / "train.csv")
train_series = pd.read_csv(DATA_DIR / "train_series.csv")
dutch_reports = train.Report.str.contains('geen|bevindingen|knie|afwijkingen')



for i in range(10):
    selected_patient = train[dutch_reports].iloc[i]
    
    print("Report")
    print(selected_patient[report_col])
    
    print("Target variables")
    print(selected_patient.drop(report_col).drop(pid_col).to_frame())

    pid = selected_patient[pid_col]
    print("StudyInstanceUID / Patient ID (pid)")
    print(pid)
    
    # Load and plot middle slice for each anatomical plane
    p_series = train_series[train_series[pid_col] == pid].drop_duplicates("Anatomical_Plane")

    print()
    print("# Patient Series Data")
    print(p_series.shape)
    print(p_series)
    fig, axes = plt.subplots(1, len(p_series), figsize=(5 * len(p_series), 5))
    axes = np.atleast_1d(axes)
    
    for ax, (_, row) in zip(axes, p_series.iterrows()):
        dcm_files = sorted((DATA_DIR / "train_series" / pid / row["SeriesInstanceUID"]).glob("*.dcm"))
        
        print("len(dcm_files)")
        print(len(dcm_files))
        if dcm_files:
            ax.imshow(pydicom.dcmread(dcm_files[len(dcm_files) // 2]).pixel_array, cmap="gray")
        ax.set_title(row["Anatomical_Plane"])
        ax.axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms


# ============================================================
# RSNA KNEE ABNORMALITY DETECTION
# ============================================================

COMPETITION_PATH = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ============================================================
# LOAD COMPETITION DATA
# ============================================================

df_train = pd.read_csv(
    os.path.join(
        COMPETITION_PATH,
        "train.csv"
    )
)

df_series = pd.read_csv(
    os.path.join(
        COMPETITION_PATH,
        "train_series.csv"
    )
)

print("train:", df_train.shape)
print("series:", df_series.shape)

print("\ntrain columns:")
print(df_train.columns.tolist())

print("\nseries columns:")
print(df_series.columns.tolist())


# ============================================================
# SELECT ONE PATIENT / STUDY
# ============================================================

# Change this if you want another patient.
#
# The first StudyInstanceUID in train_series.csv is used.
# This guarantees that the ID corresponds to the actual
# DICOM directory structure.

study_uid = str(
    df_series["StudyInstanceUID"].iloc[0]
)

print("\nSelected StudyInstanceUID:")
print(study_uid)


# ============================================================
# GET ALL SERIES FOR THIS STUDY
# ============================================================

study_series = df_series[
    df_series["StudyInstanceUID"].astype(str)
    == study_uid
].copy()

print("\nSeries belonging to study:")
print(study_series)


# ============================================================
# SELECT AXIAL SERIES
# ============================================================

axial_series = study_series[
    study_series["Anatomical_Plane"]
    .astype(str)
    .str.lower()
    .eq("axial")
].copy()

if len(axial_series) == 0:

    print(
        "\nNo axial series found."
    )

    print(
        "\nAvailable series:"
    )

    print(
        study_series[
            [
                "SeriesInstanceUID",
                "Anatomical_Plane"
            ]
        ]
    )

    raise ValueError(
        f"No axial series found for {study_uid}"
    )


print("\nAxial series:")
print(
    axial_series[
        [
            "SeriesInstanceUID",
            "Anatomical_Plane"
        ]
    ]
)


# ============================================================
# SELECT FIRST AXIAL SERIES
# ============================================================

series_uid = str(
    axial_series.iloc[0]["SeriesInstanceUID"]
)

print(
    "\nSelected SeriesInstanceUID:"
)

print(series_uid)


# ============================================================
# CORRECT RSNA KNEE IMAGE PATH
#
# train_series/
#     StudyInstanceUID/
#         SeriesInstanceUID/
#             *.dcm
# ============================================================

series_dir = os.path.join(
    COMPETITION_PATH,
    "train_series",
    study_uid,
    series_uid
)

print(
    "\nSeries directory:"
)

print(series_dir)

if not os.path.isdir(series_dir):

    raise FileNotFoundError(
        f"Series directory not found:\n{series_dir}"
    )


# ============================================================
# LOAD DICOM SERIES
# ============================================================

def load_dicom_series(series_dir):

    dicoms = []

    filenames = sorted(
        os.listdir(series_dir)
    )

    for filename in filenames:

        if not filename.lower().endswith(".dcm"):
            continue

        path = os.path.join(
            series_dir,
            filename
        )

        try:

            ds = pydicom.dcmread(
                path,
                force=True
            )

            if hasattr(ds, "PixelData"):

                dicoms.append(
                    ds
                )

        except Exception as e:

            print(
                "Could not read:",
                path,
                e
            )

    if len(dicoms) == 0:

        raise ValueError(
            f"No DICOM images found in:\n{series_dir}"
        )

    # Prefer InstanceNumber for ordering.
    if all(
        hasattr(ds, "InstanceNumber")
        for ds in dicoms
    ):

        dicoms.sort(
            key=lambda ds:
            int(ds.InstanceNumber)
        )

    elif all(
        hasattr(ds, "ImagePositionPatient")
        for ds in dicoms
    ):

        dicoms.sort(
            key=lambda ds:
            float(
                ds.ImagePositionPatient[2]
            )
        )

    return dicoms


dicoms = load_dicom_series(
    series_dir
)

print(
    "\nNumber of slices:",
    len(dicoms)
)


# ============================================================
# DICOM -> PIL RGB
# ============================================================

def dicom_to_pil(ds):

    image = ds.pixel_array.astype(
        np.float32
    )

    # Apply DICOM rescaling if present.
    slope = float(
        getattr(
            ds,
            "RescaleSlope",
            1.0
        )
    )

    intercept = float(
        getattr(
            ds,
            "RescaleIntercept",
            0.0
        )
    )

    image = (
        image * slope
        + intercept
    )

    # Robust normalization.
    low, high = np.percentile(
        image,
        [1, 99]
    )

    if high > low:

        image = np.clip(
            image,
            low,
            high
        )

        image = (
            (image - low)
            / (high - low)
        )

    else:

        image = np.zeros_like(
            image
        )

    image = (
        image * 255
    ).astype(
        np.uint8
    )

    return Image.fromarray(
        image
    ).convert("RGB")


# ============================================================
# PREPROCESSING
# ============================================================

transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# RESNET-50 FEATURE EXTRACTOR
# ============================================================

print(
    "\nLoading ResNet-50..."
)

model = models.resnet50(
    weights=models.ResNet50_Weights.IMAGENET1K_V2
)

# Remove ImageNet classification layer.
#
# Output becomes:
#
#     [batch, 2048]
#
model.fc = nn.Identity()

model = model.to(
    DEVICE
)

model.eval()

print(
    "ResNet-50 loaded."
)

print(
    "Device:",
    DEVICE
)


# ============================================================
# EXTRACT FEATURES FROM ALL SLICES
# ============================================================

@torch.no_grad()
def extract_series_features(
    dicoms,
    batch_size=32
):

    tensors = []

    print(
        "\nConverting DICOM slices..."
    )

    for i, ds in enumerate(dicoms):

        image = dicom_to_pil(
            ds
        )

        tensor = transform(
            image
        )

        tensors.append(
            tensor
        )

        if (i + 1) % 50 == 0:

            print(
                f"Prepared {i + 1}/{len(dicoms)} slices"
            )

    tensors = torch.stack(
        tensors
    )

    print(
        "\nTensor shape:",
        tensors.shape
    )

    slice_features = []

    print(
        "\nRunning ResNet-50..."
    )

    for i in range(
        0,
        len(tensors),
        batch_size
    ):

        batch = tensors[
            i:i + batch_size
        ].to(DEVICE)

        features = model(
            batch
        )

        slice_features.append(
            features.cpu()
        )

        print(
            f"Processed "
            f"{min(i + batch_size, len(tensors))}"
            f"/{len(tensors)} slices"
        )

    slice_features = torch.cat(
        slice_features,
        dim=0
    )

    return slice_features.numpy()


# ============================================================
# SLICE FEATURES
# ============================================================

slice_features = extract_series_features(
    dicoms
)

print(
    "\n============================================"
)

print(
    "Slice feature matrix:",
    slice_features.shape
)

print(
    "Expected shape:",
    f"({len(dicoms)}, 2048)"
)

print(
    "============================================"
)


# ============================================================
# ONE FEATURE VECTOR FOR THE COMPLETE SERIES
# ============================================================

feature_vector = slice_features.mean(
    axis=0
).astype(
    np.float32
)

print(
    "\nSeries feature vector shape:"
)

print(
    feature_vector.shape
)

print(
    "\nFeature vector:"
)

print(
    feature_vector
)


# ============================================================
# SAVE FEATURES
# ============================================================

series_feature_file = os.path.join(
    "/kaggle/working",
    f"{study_uid}_{series_uid}_features.npy"
)

slice_feature_file = os.path.join(
    "/kaggle/working",
    f"{study_uid}_{series_uid}_slice_features.npy"
)

np.save(
    series_feature_file,
    feature_vector
)

np.save(
    slice_feature_file,
    slice_features
)

print(
    "\nSaved series feature vector:"
)

print(
    series_feature_file
)

print(
    "\nSaved per-slice feature matrix:"
)

print(
    slice_feature_file
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "\n============================================"
)

print(
    "DONE"
)

print(
    "Study:",
    study_uid
)

print(
    "Series:",
    series_uid
)

print(
    "Number of slices:",
    len(dicoms)
)

print(
    "Features per slice:",
    slice_features.shape[1]
)

print(
    "Final series vector:",
    feature_vector.shape
)

print(
    "============================================"
)